# 02 - Data Cleaning: Diabetes Hospital Readmission

**Input:** `../data/diabetic_data.csv` (raw)  
**Output:**
- `../data/cleaned.csv`: encoded and scaled, ready for modeling (notebook 04)
- `../data/cleaned_readable.csv`: same rows before encoding and scaling, with real units and labels, used for EDA (notebook 03) and Power BI

Every decision below is also written in `../docs/cleaning_log.md`.

**Workflow:**
1. Load
2. Replace `"?"` with NaN
3. Drop columns with more than 90% missing
4. Drop columns with only one value
5. Fill the remaining missing values
6. Keep one encounter per patient
7. Remove expired and hospice discharges
8. Remove invalid gender rows
9. Treat outliers with IQR
10. Group ICD-9 codes into diagnosis categories
11. Group rare medical specialties
12. Create new features
13. Create the target
14. Save the readable file
15. Encode categorical columns
16. Scale numeric columns
17. Save `cleaned.csv`

## Step 1: Load

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/diabetic_data.csv")

print("Starting shape:", df.shape)

Starting shape: (101766, 50)


## Step 2: Replace `"?"` with NaN

This file uses `"?"` for unknown values. We turn them into real `NaN` so pandas can count and fill them.

In [2]:
df = df.replace("?", np.nan)

# check: no "?" left anywhere
print('"?" left:', (df == "?").sum().sum())

"?" left: 0


## Step 3: Drop Columns with More Than 90% Missing

In [3]:
# missing % per column
missing_pct = (df.isna().mean() * 100).round(2)
missing_pct[missing_pct > 0].sort_values(ascending=False)

weight               96.86
max_glu_serum        94.75
A1Cresult            83.28
medical_specialty    49.08
payer_code           39.56
race                  2.23
diag_3                1.40
diag_2                0.35
diag_1                0.02
dtype: float64

In [4]:
# columns above the 90% line
cols_over_90 = missing_pct[missing_pct > 90].index.tolist()
print("Dropping:", cols_over_90)

df = df.drop(columns=cols_over_90)
print("Shape now:", df.shape)

Dropping: ['weight', 'max_glu_serum']
Shape now: (101766, 48)


`weight` is almost never recorded. `max_glu_serum` is missing because the test was usually not done. With less than 10% of rows filled, neither column can tell us much.

## Step 4: Drop Columns with Only One Value

A column where every row has the same value cannot help explain readmission.

In [5]:
# number of unique values per column, smallest first
df.nunique().sort_values().head(5)

examide          1
citoglipton      1
acetohexamide    2
tolbutamide      2
diabetesMed      2
dtype: int64

In [6]:
single_value_cols = df.columns[df.nunique() == 1].tolist()
print("Dropping:", single_value_cols)

df = df.drop(columns=single_value_cols)
print("Shape now:", df.shape)

Dropping: ['examide', 'citoglipton']
Shape now: (101766, 46)


## Step 5: Fill the Remaining Missing Values

| Column | Missing | Decision |
|--------|---------|----------|
| `A1Cresult` | ~83% | Fill with `"Not tested"`. Missing here means the test was not done, which is real information |
| `medical_specialty` | ~49% | Fill with `"Unknown"`. Too many missing to guess, too useful to drop |
| `payer_code` | ~40% | Fill with `"Unknown"` |
| `race` | ~2% | Fill with `"Unknown"`. We should not guess a person's race |
| `diag_1/2/3` | < 2% | Left for now. They become `"Other"` when we group the codes in Step 10 |

In [7]:
df["A1Cresult"] = df["A1Cresult"].fillna("Not tested")
df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")
df["payer_code"] = df["payer_code"].fillna("Unknown")
df["race"] = df["race"].fillna("Unknown")

# what is still missing (only the diagnosis columns should be left)
df.isna().sum()[df.isna().sum() > 0]

diag_1      21
diag_2     358
diag_3    1423
dtype: int64

## Step 6: Keep One Encounter per Patient

Many patients appear several times. Their rows are not independent, and a model could "remember" a patient. We keep only the **first** encounter of each patient (smallest `encounter_id`), the same choice made in the original study of this dataset.

In [8]:
print("Rows before:", len(df))
print("Unique patients:", df["patient_nbr"].nunique())

# sort so the earliest encounter comes first, then keep the first row of each patient
df = df.sort_values("encounter_id")
df = df.drop_duplicates(subset="patient_nbr", keep="first")

print("Rows after:", len(df))

Rows before: 101766
Unique patients: 71518
Rows after: 71518


## Step 7: Remove Expired and Hospice Discharges

A patient who died or went to hospice care **cannot be readmitted**. Keeping them would wrongly teach the model that these patients are "safe".

Codes from `IDS_mapping.csv`: 11 = Expired, 13 = Hospice / home, 14 = Hospice / medical facility, 19 / 20 / 21 = Expired (at home / medical facility / place unknown).

In [9]:
# read the discharge part of the mapping file (lines 11-41) to show the codes
discharge_map = pd.read_csv("../data/IDS_mapping.csv", skiprows=10, nrows=30)

expired_hospice_ids = [11, 13, 14, 19, 20, 21]
discharge_map.loc[discharge_map["discharge_disposition_id"].isin(expired_hospice_ids)]

,discharge_disposition_id,description
10,11,Expired
12,13,Hospice / home
13,14,Hospice / medical facility
18,19,"Expired at home. Medicaid only, hospice."
19,20,"Expired in a medical facility. Medicaid only, ..."
20,21,"Expired, place unknown. Medicaid only, hospice."


In [10]:
print("Rows to remove:", df["discharge_disposition_id"].isin(expired_hospice_ids).sum())

# keep only rows whose discharge code is NOT in the list (~ means NOT)
df = df.loc[~df["discharge_disposition_id"].isin(expired_hospice_ids)]

print("Rows after:", len(df))

Rows to remove: 1545
Rows after: 69973


## Step 8: Remove Invalid Gender Rows

In [11]:
print(df["gender"].value_counts())

df = df.loc[df["gender"] != "Unknown/Invalid"]

print("Rows after:", len(df))

gender
Female             37229
Male               32741
Unknown/Invalid        3
Name: count, dtype: int64
Rows after: 69970


## Step 9: Treat Outliers with IQR

IQR rule: a value is an outlier if it is above `Q3 + 1.5 * IQR`.

We **cap** outliers (pull them down to the upper limit) instead of deleting rows, so no patient is lost.

Two rules for this dataset:
- **Only the upper side is capped.** For every column except `number_diagnoses` the lower limit is 0 or below, so nothing is flagged. For `number_diagnoses` the lower limit is 1.5, which would flag patients with only 1 diagnosis. That is a real, valid value.
- **Columns with IQR = 0 are skipped.** `number_outpatient`, `number_emergency` and `number_inpatient` have Q1 = Q3 = 0, so the upper limit would be 0 and *every* patient with a past visit would be turned into 0. That would destroy the most important information.

In [12]:
numeric_cols = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
                "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]

# look at the IQR limits of each column before changing anything
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR
    print(col, "| Q1:", Q1, "| Q3:", Q3, "| IQR:", IQR, "| upper:", upper, "| above upper:", (df[col] > upper).sum())

time_in_hospital | Q1: 2.0 | Q3: 6.0 | IQR: 4.0 | upper: 12.0 | above upper: 1403
num_lab_procedures | Q1: 31.0 | Q3: 57.0 | IQR: 26.0 | upper: 96.0 | above upper: 98
num_procedures | Q1: 0.0 | Q3: 2.0 | IQR: 2.0 | upper: 5.0 | above upper: 3845
num_medications | Q1: 10.0 | Q3: 20.0 | IQR: 10.0 | upper: 35.0 | above upper: 1867
number_outpatient | Q1: 0.0 | Q3: 0.0 | IQR: 0.0 | upper: 0.0 | above upper: 9119
number_emergency | Q1: 0.0 | Q3: 0.0 | IQR: 0.0 | upper: 0.0 | above upper: 5100
number_inpatient | Q1: 0.0 | Q3: 0.0 | IQR: 0.0 | upper: 0.0 | above upper: 8191
number_diagnoses | Q1: 6.0 | Q3: 9.0 | IQR: 3.0 | upper: 13.5 | above upper: 42


In [13]:
# make the columns float first, because an upper limit like 13.5 is not a whole number
df[numeric_cols] = df[numeric_cols].astype(float)

# cap only the columns where IQR is bigger than 0
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR

    if IQR == 0:
        print("Skipped (IQR = 0):", col)
    else:
        n_capped = (df[col] > upper).sum()
        df.loc[df[col] > upper, col] = upper
        print("Capped", col, "at", upper, "->", n_capped, "rows")

Capped time_in_hospital at 12.0 -> 1403 rows
Capped num_lab_procedures at 96.0 -> 98 rows
Capped num_procedures at 5.0 -> 3845 rows
Capped num_medications at 35.0 -> 1867 rows
Skipped (IQR = 0): number_outpatient
Skipped (IQR = 0): number_emergency
Skipped (IQR = 0): number_inpatient
Capped number_diagnoses at 13.5 -> 42 rows


In [14]:
# check the new maximums
df[numeric_cols].describe().loc[["min", "max"]]

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
min,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
max,12.0,96.0,5.0,35.0,42.0,42.0,12.0,13.5


## Step 10: Group ICD-9 Codes into Diagnosis Categories

`diag_1`, `diag_2` and `diag_3` hold ~700-800 different ICD-9 codes each. That is far too many categories to use directly. We group them into 9 categories using the ICD-9 chapters (the same grouping as the original study, Strack et al. 2014):

| Group | ICD-9 codes |
|-------|-------------|
| Circulatory | 390-459, 785 |
| Respiratory | 460-519, 786 |
| Digestive | 520-579, 787 |
| Diabetes | 250.xx |
| Injury | 800-999 |
| Musculoskeletal | 710-739 |
| Genitourinary | 580-629, 788 |
| Neoplasms | 140-239 |
| Other | everything else, codes starting with V or E, and missing codes |

In [15]:
# turn one ICD-9 code into its group name
def icd9_group(code):
    # missing code
    if pd.isna(code):
        return "Other"
    # V and E codes are supplementary codes, not diseases
    if code.startswith("V") or code.startswith("E"):
        return "Other"
    # diabetes codes are 250.xx
    if code.startswith("250"):
        return "Diabetes"

    num = float(code)
    if (390 <= num < 460) or int(num) == 785:
        return "Circulatory"
    if (460 <= num < 520) or int(num) == 786:
        return "Respiratory"
    if (520 <= num < 580) or int(num) == 787:
        return "Digestive"
    if 800 <= num < 1000:
        return "Injury"
    if 710 <= num < 740:
        return "Musculoskeletal"
    if (580 <= num < 630) or int(num) == 788:
        return "Genitourinary"
    if 140 <= num < 240:
        return "Neoplasms"
    return "Other"

In [16]:
# apply the function to the 3 diagnosis columns -> 3 new group columns
df["diag_1_group"] = df["diag_1"].apply(icd9_group)
df["diag_2_group"] = df["diag_2"].apply(icd9_group)
df["diag_3_group"] = df["diag_3"].apply(icd9_group)

# the raw codes are no longer needed
df = df.drop(columns=["diag_1", "diag_2", "diag_3"])

df["diag_1_group"].value_counts()

diag_1_group
Circulatory        21383
Other              12132
Respiratory         9486
Digestive           6487
Diabetes            5748
Injury              4692
Musculoskeletal     4064
Genitourinary       3440
Neoplasms           2538
Name: count, dtype: int64

## Step 11: Group Rare Medical Specialties

`medical_specialty` has 70+ values and most of them are rare. We keep the 10 most common (not counting `Unknown`) and put the rest into `"Other"`.

In [17]:
print("Unique specialties before:", df["medical_specialty"].nunique())

# the 10 most common real specialties
top_10 = df.loc[df["medical_specialty"] != "Unknown", "medical_specialty"].value_counts().head(10).index.tolist()
print("Kept:", top_10)

# everything not in the top 10 and not Unknown becomes Other
df.loc[~df["medical_specialty"].isin(top_10 + ["Unknown"]), "medical_specialty"] = "Other"

print("Unique specialties after:", df["medical_specialty"].nunique())

Unique specialties before: 71
Kept: ['InternalMedicine', 'Family/GeneralPractice', 'Emergency/Trauma', 'Cardiology', 'Surgery-General', 'Orthopedics', 'Orthopedics-Reconstructive', 'Radiologist', 'Nephrology', 'Pulmonology']
Unique specialties after: 12


## Step 12: Create New Features

| New feature | How | Why |
|-------------|-----|-----|
| `total_visits` | outpatient + emergency + inpatient visits | one number for how much hospital care the patient needed in the past year |
| `age_mid` | middle of the age band, e.g. `[70-80)` -> 75 | turns the text band into a number we can scale and test |
| `num_active_meds` | number of the diabetes medicines the patient is taking (not `No`) | how complex the treatment is |
| `num_med_changes` | number of medicines whose dose went `Up` or `Down` | how much the treatment was changed during the stay |

In [18]:
# 1. total visits in the year before this stay
df["total_visits"] = df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]

# 2. age band -> middle number: "[70-80)" -> take "70" -> 70 + 5 = 75
df["age_mid"] = df["age"].str[1:].str.split("-").str[0].astype(int) + 5

# the medicine columns: everything between A1Cresult and change in the table
med_cols = df.loc[:, "metformin":"metformin-pioglitazone"].columns.tolist()
print(len(med_cols), "medicine columns:", med_cols)

21 medicine columns: ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


In [19]:
# 3. how many medicines are being taken (value is not "No")
df["num_active_meds"] = (df[med_cols] != "No").sum(axis=1)

# 4. how many medicines had their dose changed (Up or Down)
df["num_med_changes"] = df[med_cols].isin(["Up", "Down"]).sum(axis=1)

df[["age", "age_mid", "total_visits", "num_active_meds", "num_med_changes"]].head()

,age,age_mid,total_visits,num_active_meds,num_med_changes
8,[80-90),85,0.0,2,0
9,[90-100),95,0.0,2,0
4,[40-50),45,0.0,2,0
10,[40-50),45,0.0,1,0
5,[50-60),55,0.0,1,0


In [20]:
df[["total_visits", "age_mid", "num_active_meds", "num_med_changes"]].describe()

,total_visits,age_mid,num_active_meds,num_med_changes
count,69970.000000,69970.000000,69970.000000,69970.000000
mean,0.559726,65.440046,1.190124,0.261884
std,1.427786,15.974203,0.942529,0.475849
min,0.000000,5.000000,0.000000,0.000000
25%,0.000000,55.000000,1.000000,0.000000
50%,0.000000,65.000000,1.000000,0.000000
75%,1.000000,75.000000,2.000000,0.000000
max,49.000000,95.000000,6.000000,4.000000


## Step 13: Create the Target

1 = readmitted within 30 days (`<30`), 0 = not readmitted or readmitted after 30 days.

In [21]:
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)

print(df["readmitted_30"].value_counts())
print((df["readmitted_30"].value_counts(normalize=True) * 100).round(2))

readmitted_30
0    63693
1     6277
Name: count, dtype: int64
readmitted_30
0    91.03
1     8.97
Name: proportion, dtype: float64


## Step 14: Save the Readable File

Before encoding and scaling we save a copy with real units (days, ages) and text labels. EDA and Power BI need this version, because a scaled value like `-0.43` days cannot be explained to a reader.

In [22]:
# final check: no missing values
print("Missing values:", df.isna().sum().sum())
print("Shape:", df.shape)

df.to_csv("../data/cleaned_readable.csv", index=False)
print("Saved ../data/cleaned_readable.csv")

Missing values: 0
Shape: (69970, 51)


Saved ../data/cleaned_readable.csv


## Step 15: Encode Categorical Columns

| Type | Columns | Method |
|------|---------|--------|
| Yes / No columns | `gender`, `change`, `diabetesMed` | **label encoding** to 0 / 1 |
| Medicine columns (21) | `metformin` ... | 0 = not taken, 1 = taken (`Steady`, `Up` or `Down`). Up / Down is already counted in `num_med_changes` |
| Categories with no order | `race`, `payer_code`, `medical_specialty`, `A1Cresult`, the 3 diagnosis groups, admission type, discharge, admission source | **one-hot encoding** (`drop_first=True` removes one column per group to avoid duplicate information) |

The ID columns, the text `age` (replaced by `age_mid`) and the text `readmitted` (replaced by `readmitted_30`) are dropped.

In [23]:
# start from a copy so the readable df stays as it is
model_df = df.copy()

# drop IDs and columns that were replaced by new ones
model_df = model_df.drop(columns=["encounter_id", "patient_nbr", "age", "readmitted"])

# label encoding for the 2-value columns
model_df["gender"] = (model_df["gender"] == "Male").astype(int)        # 1 = Male, 0 = Female
model_df["change"] = (model_df["change"] == "Ch").astype(int)          # 1 = medicine changed
model_df["diabetesMed"] = (model_df["diabetesMed"] == "Yes").astype(int)  # 1 = on diabetes medicine

# medicine columns: 1 = taking it, 0 = not
for col in med_cols:
    model_df[col] = (model_df[col] != "No").astype(int)

model_df[["gender", "change", "diabetesMed", "metformin", "insulin"]].head()

,gender,change,diabetesMed,metformin,insulin
8,0,1,1,0,1
9,0,1,1,0,1
4,1,1,1,0,1
10,0,0,1,0,1
5,1,0,1,0,1


In [24]:
# ID codes are numbers but mean categories -> turn them into text first so get_dummies treats them as categories
for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    model_df[col] = model_df[col].astype(str)

one_hot_cols = ["race", "payer_code", "medical_specialty", "A1Cresult",
                "diag_1_group", "diag_2_group", "diag_3_group",
                "admission_type_id", "discharge_disposition_id", "admission_source_id"]

model_df = pd.get_dummies(model_df, columns=one_hot_cols, drop_first=True, dtype=int)

print("Shape after encoding:", model_df.shape)

Shape after encoding: (69970, 140)


## Step 16: Scale Numeric Columns

Numeric columns have very different ranges (e.g. `num_lab_procedures` up to 96, `num_procedures` up to 5). Logistic regression works better when they are on the same scale. `StandardScaler` changes each column to mean 0 and standard deviation 1.

The 0/1 columns and the target are **not** scaled.

In [25]:
scale_cols = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
              "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses",
              "total_visits", "age_mid", "num_active_meds", "num_med_changes"]

scaler = StandardScaler()
model_df[scale_cols] = scaler.fit_transform(model_df[scale_cols])

# check: mean should be ~0 and std ~1
model_df[scale_cols].describe().loc[["mean", "std"]].round(2)

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,total_visits,age_mid,num_active_meds,num_med_changes
mean,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Step 17: Save `cleaned.csv`

In [26]:
print("Missing values:", model_df.isna().sum().sum())
print("Final shape:", model_df.shape)
print("Target rate (%):", round(model_df["readmitted_30"].mean() * 100, 2))

model_df.to_csv("../data/cleaned.csv", index=False)
print("Saved ../data/cleaned.csv")

Missing values: 0
Final shape: (69970, 140)
Target rate (%): 8.97


Saved ../data/cleaned.csv


## Summary

| Stage | Rows |
|-------|------|
| Raw data | 101,766 |
| One encounter per patient | 71,518 |
| Expired / hospice removed | 69,973 |
| Invalid gender removed | **69,970** |

See `../docs/cleaning_log.md` for every decision and its reason.